# Get to Know a Dataset: TAIR Multimodal Imaging Dataset for Cerebrovascular Disease

**AWS Open Data Sponsorship Program — draft tutorial**  
**Audience:** medical imaging researchers, engineers, and students  
**Duration:** 45–75 minutes  

This hands-on notebook demonstrates how to discover a public TAIR S3 dataset, inspect its contents, download a small example, explore DICOM metadata, and visualize an image or fluoroscopy frame. It ends with a reproducible community challenge.

> **Before publishing:** replace the clearly marked configuration placeholders with the verified public S3 bucket, prefix, official documentation, license, citation, and a known-safe sample object. No bucket name or study counts are assumed here. This notebook does not contain patient data.

**Research-use notice:** Imaging data are not for clinical diagnosis. Respect dataset terms, patient privacy, and institutional requirements.

## 1. Dataset overview and research questions

TAIR is described as a multimodal cerebrovascular imaging dataset including CTA/MRA and fluoroscopy. Verify and document the released modalities, cohort composition, acquisition sites, annotations, de-identification approach, and splits against the official dataset documentation before publishing.

**Example research questions**
- How can researchers access selected studies without downloading the entire dataset?
- What image dimensions, modality codes, and acquisition metadata are present?
- How might CTA/MRA and fluoroscopy support vessel visualization, segmentation, or multimodal research?

**Complete these publication details:**

| Field | Verified value to insert |
|---|---|
| Dataset homepage | TODO |
| AWS Registry entry | TODO |
| S3 URI | TODO |
| Version / release date | TODO |
| License | TODO — verify actual release terms |
| Citation / DOI | TODO |
| Dataset size / cohort | TODO — verify before publication |
| De-identification statement | TODO — link to official documentation |


## 2. Environment and costs

Python 3.10+ recommended. The tutorial uses `boto3`, `botocore`, `pydicom`, `numpy`, and `matplotlib`; compressed DICOM pixel data may require additional codecs such as `pylibjpeg` and plugins. A public bucket can generally be listed/read anonymously, but your compute, notebook runtime, and some network transfers may incur costs. This notebook limits listing and downloads by default.

Install dependencies in your notebook environment if necessary:

In [ ]:
# Uncomment and run if dependencies are missing:
# %pip install boto3 botocore pydicom numpy matplotlib pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

In [ ]:
from pathlib import Path
import os
import re
import tempfile
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import numpy as np
import matplotlib.pyplot as plt
import pydicom

print('Ready: boto3 / numpy / matplotlib / pydicom')

## 3. Configure the public dataset location

Set the verified bucket and optional prefix. Avoid guessing names or using private buckets. Anonymous access is intentional; no AWS credentials are required for a correctly configured public dataset.

In [ ]:
BUCKET = os.environ.get('TAIR_S3_BUCKET', 'multimodal-imaging-for-cerebrovascular-disease')
PREFIX = os.environ.get('TAIR_S3_PREFIX', '')  # e.g. a published 'release/' prefix
MAX_KEYS = 100  # avoid scanning the entire bucket
MAX_DOWNLOAD_MB = 100  # prevent accidental large downloads

if BUCKET == 'REPLACE_WITH_PUBLIC_TAIR_BUCKET':
    print('SETUP REQUIRED: set TAIR_S3_BUCKET or edit BUCKET to the verified public bucket.')

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED, retries={'max_attempts': 3}))

## 4. Discover the bucket layout

Use a bounded listing to understand the public directory organization. This does **not** enumerate the full dataset.

In [ ]:
objects = []
if BUCKET != 'REPLACE_WITH_PUBLIC_TAIR_BUCKET':
    try:
        response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=MAX_KEYS)
        objects = response.get('Contents', [])
        for item in objects[:30]:
            print(f"{item['Size']/1024**2:8.2f} MiB  {item['Key']}")
        print(f'Listed {len(objects)} objects; truncated={response.get("IsTruncated", False)}')
    except Exception as exc:
        print(f'Could not list objects: {type(exc).__name__}: {exc}')
        print('Check the bucket, prefix, public permissions, and network connectivity.')
else:
    print('Skipped listing until a verified public bucket is configured.')

## 5. Select a small, public DICOM example

Prefer an officially documented demonstration object. If none is supplied, this cell considers only the bounded listing above, never downloads arbitrary large objects, and does not assume `.dcm` is the only DICOM filename convention. Set `SAMPLE_KEY` explicitly for reliable publication.

In [ ]:
SAMPLE_KEY = os.environ.get('TAIR_SAMPLE_KEY', '')  # recommended: known, verified DICOM key
candidate = None
if SAMPLE_KEY:
    candidate = next((o for o in objects if o['Key'] == SAMPLE_KEY), None)
else:
    candidate = next((o for o in objects if o['Key'].lower().endswith('.dcm')
                      and 0 < o['Size'] <= MAX_DOWNLOAD_MB * 1024**2), None)
    if candidate:
        SAMPLE_KEY = candidate['Key']

if SAMPLE_KEY:
    print('Selected key:', SAMPLE_KEY)
    if candidate:
        print('Listed size (MiB):', round(candidate['Size']/1024**2, 2))
else:
    print('No sample selected. Set TAIR_SAMPLE_KEY to a documented, small public DICOM object.')

In [ ]:
sample_path = None
if BUCKET != 'REPLACE_WITH_PUBLIC_TAIR_BUCKET' and SAMPLE_KEY:
    try:
        meta = s3.head_object(Bucket=BUCKET, Key=SAMPLE_KEY)
        size = meta['ContentLength']
        if size > MAX_DOWNLOAD_MB * 1024**2:
            print(f'Skipped: object is {size/1024**2:.1f} MiB, over safety limit.')
        else:
            sample_path = Path(tempfile.gettempdir()) / 'tair_tutorial_sample.dcm'
            s3.download_file(BUCKET, SAMPLE_KEY, str(sample_path))
            print('Downloaded', sample_path, f'({size/1024**2:.2f} MiB)')
    except Exception as exc:
        print(f'Download failed: {type(exc).__name__}: {exc}')
else:
    print('Skipped download until bucket and sample key are configured.')

## 6. Read safe DICOM metadata

Do not print names, IDs, dates of birth, accession numbers, private tags, or full DICOM headers. Even public datasets should be handled with privacy-aware defaults.

In [ ]:
ds = None
if sample_path and sample_path.exists():
    try:
        ds = pydicom.dcmread(sample_path)
        safe_fields = ['Modality', 'Rows', 'Columns', 'NumberOfFrames',
                       'BitsAllocated', 'PhotometricInterpretation', 'TransferSyntaxUID']
        for field in safe_fields:
            value = getattr(ds.file_meta, field, None) if field == 'TransferSyntaxUID' else getattr(ds, field, None)
            print(f'{field}: {value}')
    except Exception as exc:
        print(f'Could not parse DICOM: {type(exc).__name__}: {exc}')
else:
    print('No local DICOM sample yet.')

## 7. Visualize an image or fluoroscopy frame

For multiframe objects, this example displays the middle frame. For single-frame images, it displays the image. It uses percentile scaling for visualization only; it is **not** a clinically calibrated display pipeline. DICOM transfer syntax support varies by installation.

In [ ]:
if ds is not None:
    try:
        arr = ds.pixel_array
        frame = arr[len(arr)//2] if arr.ndim >= 3 and getattr(ds, 'NumberOfFrames', None) else arr
        if frame.ndim == 3 and frame.shape[-1] in (3, 4):
            display_image = frame[..., :3]
            plt.imshow(display_image)
        elif frame.ndim == 2:
            frame = frame.astype(float)
            lo, hi = np.percentile(frame[np.isfinite(frame)], [1, 99])
            plt.imshow(frame, cmap='gray', vmin=lo, vmax=hi)
        else:
            raise ValueError(f'Unsupported image shape: {frame.shape}')
        plt.title('TAIR sample — research visualization only')
        plt.axis('off')
        plt.show()
    except Exception as exc:
        print(f'Visualization unavailable: {type(exc).__name__}: {exc}')
        print('Compressed images may require pylibjpeg codecs; check pixel layout and transfer syntax.')
else:
    print('Visualization skipped: no DICOM sample loaded.')

## 8. Example summary and reproducibility

Record the **dataset version**, public S3 object key, software versions, and analysis settings. Do not redistribute any sensitive metadata or infer clinical outcomes from a single example.

In [ ]:
import importlib.metadata as metadata
for package in ['boto3', 'pydicom', 'numpy', 'matplotlib']:
    print(package, metadata.version(package))
print('Public sample S3 URI:', f's3://{BUCKET}/{SAMPLE_KEY}' if SAMPLE_KEY and BUCKET != 'REPLACE_WITH_PUBLIC_TAIR_BUCKET' else 'not configured')

## 9. Community challenge: reproducible cerebrovascular visualization

**Goal:** Using a documented public TAIR subset, create a reproducible notebook that characterizes the image data and produces a meaningful visualization or baseline segmentation.

**Suggested tasks**
1. Select a small, well-defined CTA/MRA or fluoroscopy subset and document selection criteria.
2. Summarize modality, frame count, image dimensions, and available non-identifying acquisition metadata.
3. Create at least one visualization with stated windowing/normalization.
4. *(Optional)* Develop a simple vessel-enhancement or segmentation baseline and document limitations; do not present it as clinically validated.
5. Report dataset version, code dependencies, computational cost, and appropriate citation.

**Success criteria:** another researcher can reproduce the result from public data without access to private credentials or patient identifiers.

**Responsible use:** evaluate demographic and acquisition bias, avoid patient re-identification, and follow the dataset license and institutional review requirements.

## 10. Where to go next

- Official dataset documentation: **TODO — insert verified TAIR link**
- Dataset citation and license: **TODO — insert exact citation and license**
- AWS Registry listing: **TODO — insert published Registry URL**
- AWS Open Data Sponsorship Program: https://aws.amazon.com/opendata/open-data-sponsorship-program/
- AWS tutorial onboarding guidance: https://assets.opendata.aws/aws-onboarding-handbook-for-data-providers.pdf
- AWS example notebook template: https://github.com/awslabs/open-data-docs/tree/main/get-to-know-a-dataset

### Publication checklist

- [ ] Confirm official dataset name, description, counts, modalities, and release date
- [ ] Confirm public S3 bucket/prefix and choose an actual example object
- [ ] Confirm license, citation, and de-identification wording
- [ ] Run all cells against the public release and verify output
- [ ] Compare headings and questions against the latest AWS official notebook template
- [ ] Publish notebook to a public repository and link it in the Registry metadata
